<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Frequency Domain — Theory</b></h1>
</div>

## Theoretical Foundations

This notebook documents the mathematical model, estimation methods, numerical considerations, diagnostics, and limitations used in the laboratory.
### Technical Context

The Fourier transform decomposes spatial image structure into frequency components whose magnitude and phase encode different aspects of the image.

### Core Frequency-Domain Model

For image $f(x,y)$, the 2-D DFT produces $F(u,v)$. Filtering multiplies the spectrum by a transfer function $H(u,v)$, then reconstructs with the inverse transform: $g=\mathcal{F}^{-1}\{HF\}$.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $f(x,y)$ | spatial-domain image |
| $F(u,v)$ | 2-D Fourier transform |
| $H(u,v)$ | frequency-domain filter |
| $G(u,v)$ | filtered spectrum |
| $D(u,v)$ | radial frequency distance |
| $D_0$ | cutoff frequency |

### Analytical Scope

Interpret spectra, distinguish magnitude/phase, design common frequency filters, explain ringing, remove periodic noise with notches, correct slowly varying illumination, and validate reconstruction/filtering.


## 1. Spatial-Frequency Characterization

A sinusoidal brightness pattern can be written as:

$$
g(x)=A\sin(2\pi f x+\phi)
$$

where:

- $A$ = amplitude;
- $f$ = spatial frequency;
- $\phi$ = phase.

Low spatial frequency means intensity changes slowly across space.  
High spatial frequency means intensity changes rapidly.

**Important:** high frequency does not mean high brightness.

### Deeper understanding

A 2-D sinusoidal image can be written as

$$
f(x,y)=A\cos\left(2\pi(u_0x+v_0y)+\phi\right).
$$

The vector $(u_0,v_0)$ determines both frequency and orientation. Its magnitude controls how rapidly intensity changes, while its direction is normal to the visible stripe orientation. This is why oriented texture produces energy at correspondingly oriented frequency coordinates.


## 2. 1-D DFT Validation with Synthetic Sinusoids

The DFT of a 1-D signal is:

$$
X[k]
=
\sum_{n=0}^{N-1}
x[n]e^{-j2\pi kn/N}
$$

Inverse:

$$
x[n]
=
\frac{1}{N}
\sum_{k=0}^{N-1}
X[k]e^{j2\pi kn/N}
$$

Euler's identity:

$$
e^{j\theta}
=
\cos(\theta)+j\sin(\theta)
$$

For $X=a+jb$:

$$
|X|=\sqrt{a^2+b^2}
$$

and

$$
\phi=\operatorname{atan2}(b,a)
$$

Magnitude = frequency strength.  
Phase = spatial alignment.

### Deeper understanding

For a length-$N$ sequence,

$$
X[k]=\sum_{n=0}^{N-1}x[n]e^{-j2\pi kn/N},
\qquad
x[n]=\frac{1}{N}\sum_{k=0}^{N-1}X[k]e^{j2\pi kn/N}.
$$

A real sinusoid produces conjugate-symmetric peaks at positive and negative frequencies. The discrete frequency spacing is $1/N$ cycles/sample, so spectral peak locations should be interpreted relative to the sampling interval rather than as arbitrary FFT indices.


## 3. The 2-D Fourier Transform for Images

For image $f(x,y)$:

$$
F(u,v)
=
\sum_{x=0}^{M-1}
\sum_{y=0}^{N-1}
f(x,y)
e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

The FFT computes the DFT efficiently.

`fftshift` moves the zero-frequency component to the center:

- center → low frequencies;
- farther from center → high frequencies.

Before inverse FFT, undo the shift with `ifftshift`.

### Deeper understanding

For an $M\times N$ image,

$$
F(u,v)=
\sum_{x=0}^{M-1}
\sum_{y=0}^{N-1}
f(x,y)
e^{-j2\pi(ux/M+vy/N)}.
$$

The inverse transform is

$$
f(x,y)=
\frac{1}{MN}
\sum_{u=0}^{M-1}
\sum_{v=0}^{N-1}
F(u,v)
e^{j2\pi(ux/M+vy/N)}.
$$

The coefficient $F(0,0)$ is the DC term and is proportional to the image mean. The shift operation does not change the spectrum; it only rearranges indices so the DC component appears at the center for interpretation and radial filter design.


## 4. 2-D Spectrum Interpretation

A useful orientation rule:

> Spatial stripes produce spectral energy perpendicular to the stripe direction.

Let's prove it visually.

### Deeper understanding

A strong line or repetitive structure in the spatial domain produces structured energy in the spectrum. Broad smooth regions concentrate energy near the origin, while sharp edges distribute energy over a wider frequency range.

For real-valued images,

$$
F(-u,-v)=F^*(u,v),
$$

so magnitude is centrosymmetric. This conjugate symmetry is important when editing spectral coefficients: modifying only one member of a conjugate pair can produce a complex-valued inverse transform.


## 5. Inverse FFT and Reconstruction

The Fourier transform is reversible if we keep all coefficients.

### Deeper understanding

Perfect reconstruction requires preserving the complex Fourier coefficients and reversing any index shift before the inverse transform. With consistent conventions,

$$
\mathcal{F}^{-1}\{\mathcal{F}\{f\}\}=f
$$

up to floating-point round-off. A round-trip reconstruction test is therefore a fundamental validation of the transform pipeline.


## 6. Magnitude–Phase Analysis

Every Fourier coefficient can be written as:

$$
F(u,v)=|F(u,v)|e^{j\phi(u,v)}
$$

Magnitude tells us how strong a frequency is.  
Phase strongly controls spatial organization.

### Deeper understanding

The complex spectrum can be written as

$$
F(u,v)=|F(u,v)|e^{j\phi(u,v)}.
$$

Magnitude determines the strength of each sinusoidal component; phase determines how those components align spatially. Translation illustrates the distinction clearly: shifting an image changes phase systematically while leaving magnitude unchanged. This is why phase often carries a large fraction of recognizable spatial organization.


## 7. Frequency-Domain Filtering

Let $F$ be the image spectrum and $H$ the filter:

$$
G(u,v)=H(u,v)F(u,v)
$$

Then:

$$
g(x,y)=\mathcal{F}^{-1}\{G(u,v)\}
$$

Workflow:

1. FFT;
2. center with `fftshift`;
3. construct $H$;
4. multiply $H\cdot F$;
5. undo shift;
6. IFFT;
7. keep the real component.


## 8. Frequency Distance Grid

For circular filters:

$$
D(u,v)=
\sqrt{(u-u_0)^2+(v-v_0)^2}
$$

### Deeper understanding

For a centered spectrum of size $M\times N$, a radial distance grid is commonly defined as

$$
D(u,v)=\sqrt{(u-u_c)^2+(v-v_c)^2},
$$

with $(u_c,v_c)$ at the centered DC location. A transfer function $H(D)$ is isotropic because it depends only on radial distance, not direction. This is appropriate for direction-independent smoothing but not for anisotropic or orientation-selective filtering.


## 9. Ideal, Gaussian, and Butterworth Low-Pass Filters

### Ideal LPF

$$
H(u,v)=
\begin{cases}
1,&D(u,v)\le D_0\\
0,&D(u,v)>D_0
\end{cases}
$$

### Gaussian LPF

$$
H(u,v)
=
\exp\left(
-\frac{D(u,v)^2}{2D_0^2}
\right)
$$

### Butterworth LPF

$$
H(u,v)
=
\frac{1}
{1+\left(\frac{D(u,v)}{D_0}\right)^{2n}}
$$

Butterworth order $n$ controls transition steepness.

### Deeper understanding

The three common radial low-pass families are:

$$
H_{\text{ideal}}(D)=
\begin{cases}
1,&D\le D_0\\
0,&D>D_0,
\end{cases}
$$

$$
H_{\text{Gauss}}(D)=
\exp\left(-\frac{D^2}{2D_0^2}\right),
$$

and

$$
H_{\text{Butter}}(D)=
\frac{1}{1+(D/D_0)^{2n}}.
$$

The ideal filter has maximum transition sharpness, Gaussian has the smoothest transition, and Butterworth introduces order $n$ as a tunable compromise. The spatial-domain behavior follows from these spectral transition properties.


## 10. Ringing and the Gibbs Phenomenon

A hard spectral cutoff corresponds to an oscillatory spatial response:

> abrupt spectral boundary → spatial oscillations → halos near edges

### Deeper understanding

A hard frequency cutoff corresponds to a long oscillatory impulse response in space. Convolving that response with a discontinuity produces overshoot and undershoot near edges—the Gibbs phenomenon.

The important design principle is that sharp localization in frequency generally implies broad support and oscillation in space. Smoother transfer functions reduce ringing by trading away abrupt frequency selectivity.


## 11. High-Pass Filtering

For a normalized LPF:

$$
H_{HP}=1-H_{LP}
$$

High frequencies contain edges and fine detail, but can also contain noise.

### Deeper understanding

For a normalized low-pass filter,

$$
H_{\mathrm{HP}}(u,v)=1-H_{\mathrm{LP}}(u,v).
$$

The high-pass result emphasizes rapidly changing structure and suppresses slowly varying background content. Because noise often occupies high frequencies as well, a high-pass image should not automatically be interpreted as useful detail.


## 12. High-Boost Sharpening

A pure high-pass result mainly contains detail.

For sharpening:

$$
g(x,y)=f(x,y)+k f_{HP}(x,y)
$$

### Deeper understanding

If $f_{\mathrm{HP}}$ is a high-frequency detail component, high-boost sharpening uses

$$
g=f+kf_{\mathrm{HP}}.
$$

The gain $k$ controls detail amplification. The method is equivalent in spirit to unsharp masking, but implemented through a frequency-selective decomposition. Excessive gain increases noise, halos, and clipping risk.


## 13. Convolution Theorem

$$
f*h
\quad\Longleftrightarrow\quad
F\cdot H
$$

Spatial convolution corresponds to multiplication in the frequency domain.

### Circular vs Linear Convolution

A DFT assumes periodic extension.

Therefore direct FFT multiplication naturally performs **circular convolution**.  
For ordinary linear convolution, appropriate zero-padding is generally required.


## 14. Band-Pass and Band-Reject Filters

Band-pass keeps:

$$
D_1\le D(u,v)\le D_2
$$

Band-reject removes that interval.

### Deeper understanding

A radial band-pass retains

$$
D_1\le D(u,v)\le D_2,
$$

while a band-reject suppresses the same interval. Band filters are useful when relevant structure or interference occupies an intermediate frequency range rather than only the low- or high-frequency extremes.

Sharp rectangular annuli inherit the ringing issues of ideal filters; smooth band transitions can be constructed from Gaussian or Butterworth components.


## 15. Periodic Interference Analysis

Periodic interference is one of the strongest reasons to use the frequency domain.

Repeated interference often becomes isolated off-center peaks in the spectrum.

### Deeper understanding

A periodic disturbance

$$
n(x,y)=A\cos\left(2\pi(u_0x+v_0y)+\phi\right)
$$

produces localized conjugate spectral peaks near $(u_0,v_0)$ and $(-u_0,-v_0)$. This localization is the key reason Fourier-domain methods are effective for periodic noise: the disturbance may be spread across the entire image spatially but concentrated at a few spectral coordinates.


## 16. Spectral Peak Detection

The following detector is intentionally simple:

1. remove the central low-frequency area;
2. rank remaining coefficients;
3. keep strong points separated by a minimum distance.

### Deeper understanding

Peak detection should suppress the dominant central low-frequency region before ranking candidate coefficients. A useful detector also enforces spatial separation between selected peaks so that one broad spectral lobe is not counted repeatedly.

A bright off-center coefficient is not automatically noise: legitimate repetitive texture can create strong peaks. Candidate peaks therefore require confirmation from conjugate symmetry and the corresponding spatial artifact.


## 17. Notch-Reject Filtering

A notch-reject filter suppresses a small neighborhood around selected unwanted frequencies.

Real images have conjugate-symmetric spectra, so corresponding symmetric frequencies must also be considered.

### Deeper understanding

An ideal notch-reject mask sets a small neighborhood around selected interference frequencies to zero. For real-valued reconstruction, notches should occur in conjugate-symmetric pairs.

The notch radius controls a bias–variance style trade-off: too small leaves residual interference, while too large removes useful neighboring frequencies. Smooth Gaussian-style notches can reduce spatial ringing compared with hard binary notches.


## 18. Moiré Removal

Moiré is a repeated interference pattern. It can often be easier to isolate in the Fourier domain than in the spatial domain.

### Deeper understanding

Moiré arises from interference between periodic sampling or texture structures. In the spectrum it often appears as distinct off-center peaks or clusters. Removal is therefore a localization problem: identify only the interference frequencies and suppress them without erasing nearby legitimate repetitive structure.


## 19. Low-Frequency Illumination Correction

A simple multiplicative model is:

$$
I(x,y)\approx R(x,y)L(x,y)
$$

where:

- $R$ = reflectance / useful structure;
- $L$ = slowly varying illumination.

Because illumination varies slowly, it is dominated by low frequencies.

### Deeper understanding

A common multiplicative image model is

$$
I(x,y)=R(x,y)L(x,y),
$$

where $R$ is reflectance and $L$ slowly varying illumination. Taking logarithms converts multiplication into addition:

$$
\log I=\log R+\log L.
$$

Because illumination tends to occupy lower frequencies than fine reflectance structure, low-frequency estimation or homomorphic-style filtering can reduce shading. The separation is approximate and depends on the image content.


## 20. Cutoff Sensitivity

For a low-pass filter:

- smaller cutoff → stronger smoothing;
- larger cutoff → more detail preserved.

### Deeper understanding

The cutoff $D_0$ is a model parameter, not merely a display control. For low-pass filtering, decreasing $D_0$ removes more high-frequency energy and increases smoothing; increasing it approaches the identity transform.

A principled choice is based on a sensitivity sweep that measures both numerical change and visual preservation of relevant structure.


## 21. Quantitative Checks

MSE:

$$
\mathrm{MSE}
=
\frac{1}{MN}
\sum_{x,y}
[f(x,y)-g(x,y)]^2
$$

PSNR:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

PSNR measures numerical fidelity to a reference. It is not a universal perceptual-quality metric.

### Deeper understanding

For reference $f$ and result $g$,

$$
\mathrm{MSE}=\frac{1}{MN}\sum_{x,y}(f-g)^2,
$$

$$
\mathrm{PSNR}=10\log_{10}\left(\frac{L^2}{\mathrm{MSE}}\right).
$$

These metrics quantify fidelity to a reference, not whether the filtering objective was achieved. A deliberately sharpened or interference-suppressed image can have lower PSNR than the original while still be more useful for the target task.


## 22. Validation Checks

This section develops the frequency-domain theory for validation checks.

### Deeper understanding

Frequency-domain validation should test transform round-trip error, mask shape, transfer-function bounds, finite values, conjugate-symmetry assumptions when relevant, and output existence. A filter mask that has the correct dimensions but is centered in the wrong coordinate convention can still produce a structurally incorrect result, so coordinate consistency must be validated explicitly.


## 23. Failure Modes and Diagnostic Signatures

Frequency-domain errors are often identifiable because they leave characteristic signatures in either the spectrum or the reconstructed image.

### Dynamic-Range Masking

The DC component and dominant low-frequency energy can be orders of magnitude larger than weak spectral peaks. A direct magnitude display can therefore appear almost black away from the origin even when meaningful periodic components are present. Logarithmic display compresses that dynamic range for diagnosis while leaving the actual complex spectrum unchanged for filtering.

### Shift Misalignment

A centered spectrum requires a transfer function designed around the centered frequency origin. Applying a mask in the wrong coordinate convention suppresses unintended frequencies even though the array dimensions remain valid. The error is therefore structural rather than syntactic.

### Abrupt Spectral Transitions

A hard cutoff produces a spatial response with oscillatory sidelobes. Near strong edges, those sidelobes appear as overshoot and undershoot. Ringing should therefore be diagnosed from edge profiles or overshoot measurements, not only from global image metrics.

### Sharpening and Noise

High-frequency enhancement does not distinguish useful fine structure from high-frequency acquisition noise. A sharpening configuration that increases edge contrast can simultaneously increase noise variance.

### Localized Spectral Suppression

A bright off-center spectral peak is not automatically interference. Repetitive texture can generate valid peaks. Notch placement must therefore be supported by spatial periodicity, conjugate symmetry, and before/after evidence.

### Circular-Convolution Artifacts

DFT-domain multiplication without adequate padding corresponds to periodic boundary assumptions. When the intended operation is ordinary linear convolution, insufficient padding produces wrap-around contamination near image boundaries.

### Diagnostic Principle

A valid frequency-domain result should be supported by multiple forms of evidence: spectral structure, spatial reconstruction, numerical checks, and consistency with the assumed image-formation or degradation mechanism.


## 24. Parameter Sensitivity and Controlled Experiments

Parameter selection is an experimental design problem. A meaningful sensitivity study varies one control while holding the remaining processing chain fixed.

### Cutoff Sensitivity

For low-pass filtering, decreasing the cutoff removes progressively more high-frequency structure. This generally increases smoothing and deviation from the original image. Increasing the cutoff preserves more detail but reduces the strength of the filtering intervention.

### Butterworth Order

The order controls how rapidly the transfer function changes around the cutoff. Low orders produce gradual transitions; higher orders approach a sharper boundary and therefore increase the possibility of ringing around strong spatial discontinuities.

### High-Boost Gain

The gain determines how strongly the extracted high-frequency component is added back to the image. Increasing it can improve local edge contrast while also increasing clipping and noise sensitivity.

### Notch Radius

A small notch may leave part of a periodic component untouched. An excessively large notch can remove neighboring frequencies that belong to useful image structure. Radius selection is therefore a localization trade-off.

### Controlled-Experiment Rule

For each sweep:

1. keep the input fixed;
2. keep non-target parameters fixed;
3. vary only the parameter under study;
4. record a quantitative response;
5. inspect the corresponding spatial or spectral output;
6. select a setting only after the trend is understood.

This makes the retained configuration reproducible and defensible.


## 25. Method Selection and Technical Discussion

Filter choice depends on the processing objective and on the failure modes that matter for that objective.

### Low-Pass Family

- **Ideal:** maximally sharp frequency separation, but the abrupt boundary makes ringing likely near strong edges.
- **Gaussian:** smooth transition and low ringing tendency, useful when artifact suppression and spatial smoothness are priorities.
- **Butterworth:** intermediate behavior with an explicit order parameter that controls transition steepness.

No family is universally best. The relevant comparison is between frequency selectivity, spatial artifacts, and preservation of task-relevant structure.

### Detail Enhancement

A high-pass image isolates rapidly varying content. High-boost sharpening reinjects that content into the original image, which is usually more useful for visualization than displaying the high-pass component alone. Both methods require noise checks.

### Periodic Interference

Notch rejection is appropriate when unwanted periodic components form localized spectral peaks that can be separated from useful image content. The notch position and radius must be validated against the corresponding spatial artifact.

### Illumination Variation

Slowly varying illumination is concentrated near low spatial frequencies. Frequency-domain estimation can therefore separate broad illumination trends from faster reflectance structure, but the correction model must remain consistent with the assumed image-formation process.

### Evidence-Based Selection

Method selection should combine quantitative change, edge preservation, ringing, clipping or noise behavior, and spectral localization. These criteria may favor different methods for different tasks; the final choice should state which objective is being optimized.


## 26. Integrated Frequency-Domain Workflow

A complete frequency-domain workflow links diagnosis, filter design, reconstruction, and validation rather than treating them as independent operations.

~~~text
Input image
    ↓
Input validation and spatial inspection
    ↓
2-D transform and centering
    ↓
Spectrum diagnosis
    ↓
Task-specific filter selection
    ↓
Parameter selection from controlled evidence
    ↓
Complex-spectrum filtering
    ↓
Inverse centering and reconstruction
    ↓
Spatial + spectral + numerical validation
    ↓
Configuration report and reproducible output
~~~

The workflow separates three responsibilities:

1. **Diagnosis:** determine what spectral structure corresponds to the processing objective.
2. **Intervention:** construct the smallest justified frequency-domain modification.
3. **Validation:** verify that the intended structure changed while unacceptable artifacts were not introduced.

The same execution pattern supports low-pass smoothing, high-frequency enhancement, band isolation, notch-based periodic-noise suppression, moiré reduction, and low-frequency illumination estimation.


## Technical Synthesis

Frequency-domain processing is expressed by the transfer-function formulation

$$
G(u,v)=H(u,v)F(u,v),
$$

with reconstruction through the inverse Fourier transform. The complete analysis chain is

$$
\boxed{
\text{image}
\rightarrow
\mathcal{F}
\rightarrow
\text{spectrum interpretation}
\rightarrow
H(u,v)
\rightarrow
\mathcal{F}^{-1}
\rightarrow
\text{spatial result}
\rightarrow
\text{diagnostics}
}
$$

Filter family, cutoff, order, spectral localization, phase preservation, conjugate symmetry, ringing, and periodic-noise signatures determine whether a frequency-domain intervention is justified.

## Scope and Limitations

### Included

2-D DFT/IDFT, spectrum reading, magnitude/phase, low/high/band filters, ringing, convolution theorem, periodic-noise detection/removal, moiré, shading, sensitivity, and validation.

### Not included

Wavelets and advanced multiresolution methods.


## References

1. **R. C. Gonzalez and R. E. Woods**, *Digital Image Processing* — core reference for the 2-D DFT, frequency-domain filtering, ideal/Gaussian/Butterworth filters, periodic-noise removal, and homomorphic/illumination processing. [Companion site](https://www.imageprocessingplace.com/)
2. **NumPy Documentation**, “Discrete Fourier Transform (`numpy.fft`)” — exact transform conventions and API family used for FFT/IFFT, 2-D transforms, frequency bins, and shift operations. [NumPy FFT reference](https://numpy.org/doc/stable/reference/routines.fft.html)
3. **NumPy Documentation**, `numpy.fft.fft` — implementation-level reference for DFT normalization conventions, FFT behavior, and Hermitian symmetry of real-valued inputs. [NumPy FFT API](https://numpy.org/doc/stable/reference/generated/numpy.fft.fft.html)
4. **J. W. Cooley and J. W. Tukey**, “An Algorithm for the Machine Calculation of Complex Fourier Series,” *Mathematics of Computation*, 1965 — foundational FFT algorithm reference cited by NumPy. [DOI: 10.1090/S0025-5718-1965-0178586-1](https://doi.org/10.1090/S0025-5718-1965-0178586-1)
5. **OpenCV Documentation**, “Image Filtering” — complementary reference for spatial/frequency interpretation of linear filtering and border assumptions when validating convolution-equivalent operations. [OpenCV filtering reference](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html)
